In [ ]:
import slangpy as spy
import numpy as np
import matplotlib.pyplot as plt

from bvhgs import device
from bvhgs import gaussian_module

In [ ]:
np.random.seed(0)

In [ ]:
test_module = device.load_module("tests.slang")

In [ ]:
render_prog = device.link_program([test_module], [test_module.entry_point("renderGaussian2D")])
render_ker = device.create_compute_kernel(render_prog)

In [ ]:
render_target = device.create_texture(
    type=spy.TextureType.texture_2d,
    format=spy.Format.rgba32_float,
    width=64,
    height=64,
    usage=spy.TextureUsage.shader_resource|spy.TextureUsage.unordered_access
)

In [ ]:
position = np.random.uniform(low=0.3, high=0.7, size=(3)).astype(np.float32)
covariance = np.random.randn(2, 2).astype(np.float32)
cov = np.dot(covariance, covariance.T) * 0.002
color = np.random.rand(3).astype(np.float32)
opacity = np.random.rand(1).astype(np.float32)
gaussian = gaussian_module.Gaussian2D(
    position=spy.float3(*position),
    covariance=spy.float2x2(cov.flatten()),
    color=spy.float3(*color),
    opacity=opacity.item(),
)
gaussian

In [ ]:
render_ker.dispatch(
    thread_count=[64, 64, 1],
    renderTarget=render_target,
    g=gaussian
)

In [ ]:
render_target.to_bitmap()